In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("omkargurav/face-mask-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'face-mask-dataset' dataset.
Path to dataset files: /kaggle/input/face-mask-dataset


### Import Libraries

In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

### Data Preprocessing

In [3]:
IMG_SIZE = 299
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

train_generator = train_datagen.flow_from_directory(
    "/kaggle/input/face-mask-dataset/data",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training',
    color_mode = 'rgb'
)

val_generator = train_datagen.flow_from_directory(
    "/kaggle/input/face-mask-dataset/data",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation',
    color_mode = 'rgb'
)

Found 6043 images belonging to 2 classes.
Found 1510 images belonging to 2 classes.


In [4]:
train_labels = train_generator.classes

val_labels = val_generator.classes

train_with_mask = np.sum(train_labels == 0)
train_without_mask = np.sum(train_labels == 1)

val_with_mask = np.sum(val_labels == 0)
val_without_mask = np.sum(val_labels == 1)

print("TRAIN SET:")
print("With Mask:", train_with_mask)
print("Without Mask:", train_without_mask)

print("\nVALIDATION SET:")
print("With Mask:", val_with_mask)
print("Without Mask:", val_without_mask)

TRAIN SET:
With Mask: 2980
Without Mask: 3063

VALIDATION SET:
With Mask: 745
Without Mask: 765


In [5]:
train_generator.class_indices

{'with_mask': 0, 'without_mask': 1}

### Load Inception V3 Model

In [6]:
base_model = InceptionV3(
    weights='imagenet',
    include_top=False,
    input_shape=(299, 299, 3)
)

87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


### Modify Model

In [7]:
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=output)

### Freeze Base Layers

In [8]:
for layer in base_model.layers:
    layer.trainable = False

In [9]:
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

### Train Model

In [10]:
face_mask_model = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10
)

Epoch 1/10
 42/189 ━━━━━━━━━━━━━━━━━━━━ 2:06 859ms/step - accuracy: 0.8474 - loss: 0.2719

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


189/189 ━━━━━━━━━━━━━━━━━━━━ 243s 1s/step - accuracy: 0.9767 - loss: 0.0587 - val_accuracy: 0.9887 - val_loss: 0.0253
Epoch 2/10
189/189 ━━━━━━━━━━━━━━━━━━━━ 163s 865ms/step - accuracy: 0.9919 - loss: 0.0231 - val_accuracy: 0.9940 - val_loss: 0.0179
Epoch 3/10
189/189 ━━━━━━━━━━━━━━━━━━━━ 166s 877ms/step - accuracy: 0.9894 - loss: 0.0303 - val_accuracy: 0.9722 - val_loss: 0.0809
Epoch 4/10
189/189 ━━━━━━━━━━━━━━━━━━━━ 166s 881ms/step - accuracy: 0.9949 - loss: 0.0169 - val_accuracy: 0.9940 - val_loss: 0.0226
Epoch 5/10
189/189 ━━━━━━━━━━━━━━━━━━━━ 166s 880ms/step - accuracy: 0.9906 - loss: 0.0273 - val_accuracy: 0.9947 - val_loss: 0.0214
Epoch 6/10
189/189 ━━━━━━━━━━━━━━━━━━━━ 176s 935ms/step - accuracy: 0.9957 - loss: 0.0144 - val_accuracy: 0.9940 - val_loss: 0.0214
Epoch 7/10
189/189 ━━━━━━━━━━━━━━━━━━━━ 170s 901ms/step - accuracy: 0.9960 - loss: 0.0131 - val_accuracy: 0.9921 - val_loss: 0.0267
Epoch 8/10
189/189 ━━━━━━━━━━━━━━━━━━━━ 165s 873ms/step - accuracy: 0.9949 - loss: 0.0158 

In [12]:
model.save("content/face_mask_inceptionv3.keras")